In [1]:
import datetime
import os
import warnings; warnings.filterwarnings("ignore")

import mysql.connector as conn
import numpy as np
import pandas as pd; pd.set_option("display.float_format","{:,.2f}".format)
import sqlalchemy
from dotenv import load_dotenv
from sqlalchemy.sql import text

In [2]:
# Takes environment variables from .env.
load_dotenv()

# Credentials for local PostgreSQL database.
pg_host = "localhost"
pg_database = "paid_content"
pg_schema = "user_profile"
pg_table = "activity"
pg_user = "postgres"
pg_pass = os.environ["POSTGRES_PASSWORD"]
pg_port = 5432

# Credentials for internal Delfi database.
dlf_name_articles = "delfi_articles"
dlf_name_paywall = "paywall"
dlf_host = os.environ["DELFI_DB_HOST"]
dlf_user = os.environ["DELFI_DB_USER"]
dlf_pass = os.environ["DELFI_DB_PASSWORD"]
dlf_port = os.environ["DELFI_DB_PORT"]

# MySQL connector.
delfi_paywall_conn = conn.connect(
    host=dlf_host,
    port=dlf_port,
    user=dlf_user,
    password=dlf_pass,
    database=dlf_name_paywall
)

# 90 days period

In [14]:
#start_date = "2023-01-25"
#end_date = "2023-04-24"

# Query all data about activity of paid content users from local PostgreSQL database.
postgresql_url = f"postgresql+psycopg2://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_database}"

engine = sqlalchemy.create_engine(postgresql_url)

activity_query = text(
    """
    SELECT *
    FROM user_profile.activity
    WHERE user_delfi_id NOT IN (1415556, 963890, 930767)
    AND date BETWEEN (SELECT TO_DATE('2023-01-25','YYYY-MM-DD')) AND (SELECT TO_DATE('2023-04-24','YYYY-MM-DD'))
    """
)

with engine.connect().execution_options(autocommit=True) as conn:
    activity_query_sql = conn.execute(activity_query)

initial_activity_data = pd.DataFrame(activity_query_sql.fetchall())

#initial_activity_data["activity_date"] = initial_activity_data["timestamp"].dt.date
#initial_activity_data["activity_year"] = initial_activity_data["timestamp"].dt.year
#initial_activity_data["activity_month"] = initial_activity_data["timestamp"].dt.month

unique_users = initial_activity_data["user_delfi_id"].nunique()
num_row = len(initial_activity_data)
print(f"Initial data consists of {num_row} rows and {unique_users} unique users.")

min_date = initial_activity_data["date"].min()
max_date = initial_activity_data["date"].max()
print(f"Range of date is from {min_date} to {max_date}.")
print("Duration of period (Days):", initial_activity_data["date"].nunique())

Initial data consists of 490833 rows and 20775 unique users.
Range of date is from 2023-01-25 to 2023-04-24.
Duration of period (Days): 90


Here and in the subsequent calculations of average values I use median because distribution of all data is more or less skewed to the rigth. 

In [5]:
rfv_df = initial_activity_data.groupby("user_delfi_id").agg(
    frequency=("date", lambda x: x.nunique()),
    volume=("pageviews", "sum"),
    last_visit=("date", "max"),
    avg_active_time=("avg_active_time", "median"),
    avg_scroll_depth=("avg_scroll_depth", "median")
).reset_index()

# The latest date is literally the last date of data and is used to calculate recency.
latest_date = initial_activity_data["date"].max().strftime("%Y-%m-%d")
anchor_date = pd.to_datetime(latest_date).date()
rfv_df["recency"] = (anchor_date - rfv_df["last_visit"]).dt.days

# Data with final RFV scores.
rfv_df["rfv_score"] = (rfv_df["frequency"] * np.sqrt((rfv_df["volume"]))) / (rfv_df["recency"] + 1)
rfv_df["user_delfi_id"] = rfv_df["user_delfi_id"].astype("string")
rfv_df.head()

,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score
0,1735,26,36,2023-04-23,24.50,40.00,2,52.00
1,2140,55,108,2023-04-16,226.00,56.00,9,57.16
2,2320,62,119,2023-04-25,114.00,49.00,0,676.34
3,2872,42,73,2023-04-21,55.00,67.00,4,71.77
4,3069,80,270,2023-04-25,78.50,51.00,0,"1,314.53"


In [6]:
# User ID and User Delfi ID from Delfi internal database.
user_id_query = """
    SELECT id, delfi_id
    FROM paywall.users
    WHERE delfi_id IS NOT NULL
    """

user_id_data = pd.read_sql(user_id_query, delfi_paywall_conn)
user_id_data = user_id_data.astype("string")

In [7]:
rfv_df = rfv_df.merge(user_id_data, how="left", left_on="user_delfi_id", right_on="delfi_id").rename(columns={"id": "user_id"})
rfv_df.head()

,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score,user_id,delfi_id
0,1735,26,36,2023-04-23,24.50,40.00,2,52.00,4362,1735
1,2140,55,108,2023-04-16,226.00,56.00,9,57.16,14449,2140
2,2320,62,119,2023-04-25,114.00,49.00,0,676.34,5856,2320
3,2872,42,73,2023-04-21,55.00,67.00,4,71.77,56979,2872
4,3069,80,270,2023-04-25,78.50,51.00,0,"1,314.53",56747,3069


I tried to select just those users who were active during the whole period: theirs subscriptions valid_till date was greater than the upper bound of period and first purchase date was older than the lower bound of period. With the date of first purchase I eliminate those users who bought theirs first subscriptions anytime during the period. Hence in theory all users have the equal possibilities to visit website the same number of times and read the same amount of articles.

When time period is increasing the probability that some users would churn and buy subscription again after some break (aka returned user) is also increasing. Due to constraints of our database I cannot eliminate such behavior but in our case number of such users is quite small.

In [8]:
# Users who were active during the whole period.
users_query = """
    SELECT DISTINCT(P1.user_id)
    FROM (SELECT T.user_id, T.completion_timestamp, S.plan, US.renewal,
        ROW_NUMBER() OVER (PARTITION BY T.user_id ORDER BY T.completion_timestamp) AS row_num
        FROM paywall.transactions AS T
		LEFT JOIN paywall.user_subscriptions AS US
		ON T.user_id = US.user_id
		LEFT JOIN paywall.subscriptions AS S
		ON US.subscription_id = S.id
		WHERE US.service_id IN (1, 3, 4, 9, 10)
		AND S.included_in_count = 1
		AND T.completion_timestamp IS NOT NULL
		AND S.sf_category = "Paid subscriptions"
		AND US.valid_till > "%s 23:59:59") AS P1
    WHERE P1.row_num = 1
    AND P1.completion_timestamp < "2023-01-25 00:00:00"
    """ % latest_date

users = pd.read_sql(users_query, delfi_paywall_conn)
users = users.astype("string")

In [9]:
users_final_df = rfv_df.merge(users, on="user_id").drop(columns="delfi_id")
users_final_df = users_final_df.astype({"user_delfi_id": "string", "user_id": "string"})

unique_users = users_final_df["user_delfi_id"].nunique()
print(f"Final data consists of {unique_users} unique users.")

users_final_df.head()

Final data consists of 12142 unique users.


,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score,user_id
0,2320,62,119,2023-04-25,114.00,49.00,0,676.34,5856
1,2872,42,73,2023-04-21,55.00,67.00,4,71.77,56979
2,3069,80,270,2023-04-25,78.50,51.00,0,"1,314.53",56747
3,11878,26,33,2023-04-18,111.50,74.00,7,18.67,99419
4,12519,18,22,2023-04-21,15.00,30.00,4,16.89,47013


In [10]:
#users_final_df.to_excel("agg_data.xlsx", index=False)

RFV score consist of three main metrics:
- Recency - number of days since user visited the website last time.
- Frequency - how much days user visited the website during the period.
- Volume - number of read articles (pageviews).

Since there are no correlations between RFV and other users related metrics like Time on page, Scroll depth, Count of transactions or Total amount spent, we cannot understand when RFV is good one or bad one. From business perspective we decided that users who visited our website in the last 7 days - are the good ones and others - not so good. The former we called FANS, and the latter - SLEEPERS. In the first table below it is visible that these two groups of users have very different metrics, so our theory is correct. 

In other words, we determined our threshold of recency and looking to the data we can find out average frequency and average volume when recency is equal to 6. Knowing these two metrics we can calculate RFV threshold using RFV formula.

In [10]:
recency_threshold = 6

syntetic_rfv_data = users_final_df.copy()
syntetic_rfv_data["user_type"] = np.where(users_final_df["recency"] <= recency_threshold, "fans", "sleepers")

pd.DataFrame({
    "Users": syntetic_rfv_data.groupby("user_type")["user_id"].count(),
    "Share": syntetic_rfv_data.groupby("user_type")["user_id"].count() / syntetic_rfv_data["user_id"].count() * 100,
    "RFV score": syntetic_rfv_data.groupby("user_type")["rfv_score"].median(),
    "Frequency": syntetic_rfv_data.groupby("user_type")["frequency"].median(),
    "Recency": syntetic_rfv_data.groupby("user_type")["recency"].median(),
    "Volume": syntetic_rfv_data.groupby("user_type")["volume"].median()
}).set_axis(["Fans: Recency equal or below 6", "Idlers: Recency above 6"], axis="index")

,Users,Share,RFV score,Frequency,Recency,Volume
Fans: Recency equal or below 6,9598,79.05,178.51,38.00,1.00,61.00
Idlers: Recency above 6,2544,20.95,0.94,7.00,19.00,9.00


In [21]:
syntetic_rfv_data

,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score,user_id,user_type
0,2320,62,119,2023-04-25,114.00,49.00,0,676.34,5856,fans
1,2872,42,73,2023-04-21,55.00,67.00,4,71.77,56979,fans
2,3069,80,270,2023-04-25,78.50,51.00,0,"1,314.53",56747,fans
3,11878,26,33,2023-04-18,111.50,74.00,7,18.67,99419,sleepers
4,12519,18,22,2023-04-21,15.00,30.00,4,16.89,47013,fans
...,...,...,...,...,...,...,...,...,...,...
12137,1435608,21,56,2023-04-25,64.25,54.25,0,157.15,5038,fans
12138,1437147,4,5,2023-04-25,21.50,49.50,0,8.94,145339,fans
12139,1440387,3,3,2023-04-24,NaN,NaN,1,2.60,13086,fans
12140,1453869,2,4,2023-04-24,15.00,69.00,1,2.00,130475,fans


In [11]:
frequency_threshold = syntetic_rfv_data[syntetic_rfv_data["recency"] == recency_threshold]["frequency"].median()
volume_threshold = syntetic_rfv_data[syntetic_rfv_data["recency"] == recency_threshold]["volume"].median()
print("Threshold of Frequency:", frequency_threshold)
print("Threshold of Volume:", volume_threshold)

Threshold of Frequency: 16.0
Threshold of Volume: 20.5


In [12]:
syntetic_rfv_score = (frequency_threshold * np.sqrt((volume_threshold))) / (recency_threshold + 1)
print("Threshold of RFV score:", syntetic_rfv_score)
print("Duration of period (Days):", initial_activity_data["date"].nunique())

Threshold of RFV score: 10.349011586442762
Duration of period (Days): 91


## RFV by recency

In [13]:
recency_10 = users_final_df[users_final_df["recency"] < 11].copy()

pd.DataFrame({
    "Users": recency_10.groupby("recency")["user_id"].count(),
    "Share": recency_10.groupby("recency")["user_id"].count() / users_final_df["user_id"].count() * 100,
    "RFV score": recency_10.groupby("recency")["rfv_score"].median(),
    "Frequency": recency_10.groupby("recency")["frequency"].median(),
    "Recency": recency_10.groupby("recency")["recency"].median(),
    "Volume": recency_10.groupby("recency")["volume"].median()
}).set_axis(["Recency: 0", "Recency: 1", "Recency: 2", "Recency: 3", "Recency: 4", "Recency: 5",
             "Recency: 6", "Recency: 7", "Recency: 8", "Recency: 9", "Recency: 10"], axis="index")

,Users,Share,RFV score,Frequency,Recency,Volume
Recency: 0,4526,37.28,473.28,50.00,0.00,90.00
Recency: 1,2094,17.25,141.96,37.00,1.00,59.00
Recency: 2,1528,12.58,60.83,28.00,2.00,40.00
Recency: 3,468,3.85,30.29,22.00,3.00,29.00
Recency: 4,387,3.19,19.60,19.00,4.00,25.00
Recency: 5,353,2.91,13.88,18.00,5.00,22.00
Recency: 6,242,1.99,10.25,16.00,6.00,20.50
Recency: 7,212,1.75,9.77,17.00,7.00,21.50
Recency: 8,182,1.50,7.87,16.00,8.00,20.00
Recency: 9,200,1.65,3.98,10.00,9.00,13.00


## Sleepers

In [14]:
sleepers_rfv_data = syntetic_rfv_data[syntetic_rfv_data["user_type"] == "sleepers"]
sleepers_rfv_data.head()

,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score,user_id,user_type
3,11878,26,33,2023-04-18,111.50,74.00,7,18.67,99419,sleepers
25,80933,21,29,2023-03-19,51.00,48.00,37,2.98,16809,sleepers
26,83985,11,20,2023-02-14,73.00,45.00,70,0.69,75784,sleepers
31,95646,23,35,2023-04-11,111.00,70.00,14,9.07,92025,sleepers
32,97448,48,90,2023-04-17,317.25,81.50,8,50.60,40166,sleepers


In [15]:
# List of emails of users who gave consent for newsletters.
consent_emails = pd.read_csv("content_emails.txt", header=None, skipfooter=1).rename(columns={0: "email"}).drop_duplicates()

# Querying emails of users from internal Delfi database. 
user_email_query = """
    SELECT id, email
    FROM paywall.users
    WHERE email IS NOT NULL
    """

user_email_data = pd.read_sql(user_email_query, delfi_paywall_conn)
user_email_data["id"] = user_email_data["id"].astype("str")
user_email_data = user_email_data.rename(columns={"id": "user_id"})

consent_users = consent_emails.merge(user_email_data, how="left", on="email")
consent_users["consent"] = "1"

sleepers_rfv_data = sleepers_rfv_data.merge(consent_users[["user_id", "consent"]], how="left", on="user_id").fillna(value={"consent": "0"})

In [16]:
pd.DataFrame({
    "Users": sleepers_rfv_data.groupby("consent")["user_id"].count(),
    "Share": sleepers_rfv_data.groupby("consent")["user_id"].count() / sleepers_rfv_data["user_id"].count() * 100,
    "RFV score": sleepers_rfv_data.groupby("consent")["rfv_score"].median(),
    "Frequency": sleepers_rfv_data.groupby("consent")["frequency"].median(),
    "Recency": sleepers_rfv_data.groupby("consent")["recency"].median(),
    "Volume": sleepers_rfv_data.groupby("consent")["volume"].median()
}).set_axis(["Email consent: NO", "Email consent: YES"], axis="index")

,Users,Share,RFV score,Frequency,Recency,Volume
Email consent: NO,1994,78.38,0.82,7.00,20.00,9.00
Email consent: YES,550,21.62,1.32,8.00,16.50,10.00


In [17]:
sleepers_rfv_data.groupby("consent")["user_id"].count()

consent
0    1994
1     550
Name: user_id, dtype: int64

# 28 days period

In [21]:
# Query all data about activity of paid content users from local PostgreSQL database.
postgresql_url = f"postgresql+psycopg2://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_database}"

engine = sqlalchemy.create_engine(postgresql_url)

activity_query = text("SELECT * FROM user_profile.activity WHERE user_delfi_id NOT IN (1415556, 963890, 930767) AND date > '2023-01-31' AND date < '2023-03-01'")

with engine.connect().execution_options(autocommit=True) as conn:
    activity_query_sql = conn.execute(activity_query)

initial_activity_data = pd.DataFrame(activity_query_sql.fetchall())

#initial_activity_data["activity_date"] = initial_activity_data["timestamp"].dt.date
#initial_activity_data["activity_year"] = initial_activity_data["timestamp"].dt.year
#initial_activity_data["activity_month"] = initial_activity_data["timestamp"].dt.month

unique_users = initial_activity_data["user_delfi_id"].nunique()
num_row = len(initial_activity_data)
print(f"Initial data consists of {num_row} rows and {unique_users} unique users.")

min_date = initial_activity_data["date"].min()
max_date = initial_activity_data["date"].max()
print(f"Range of date is from {min_date} to {max_date}.")

Initial data consists of 141775 rows and 13978 unique users.
Range of date is from 2023-02-01 to 2023-02-28.


In [22]:
rfv_df = initial_activity_data.groupby("user_delfi_id").agg(
    frequency=("date", lambda x: x.nunique()),
    volume=("pageviews", "sum"),
    last_visit=("date", "max"),
    avg_active_time=("avg_active_time", "median"),
    avg_scroll_depth=("avg_scroll_depth", "median")
).reset_index()

# The latest date is literally the last date of data and is used to calculate recency.
latest_date = initial_activity_data["date"].max().strftime("%Y-%m-%d")
anchor_date = pd.to_datetime(latest_date).date()
rfv_df["recency"] = (anchor_date - rfv_df["last_visit"]).dt.days

# Data with final RFV scores.
rfv_df["rfv_score"] = (rfv_df["frequency"] * np.sqrt((rfv_df["volume"]))) / (rfv_df["recency"] + 1)
rfv_df["user_delfi_id"] = rfv_df["user_delfi_id"].astype("string")
rfv_df.head()

,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score
0,1735,8,9,2023-02-26,127.50,83.50,2,8.00
1,2140,20,33,2023-02-28,258.00,55.00,0,114.89
2,2320,19,37,2023-02-28,114.50,62.50,0,115.57
3,2872,10,20,2023-02-28,158.50,89.50,0,44.72
4,3069,25,86,2023-02-28,57.00,53.75,0,231.84


In [23]:
rfv_df = rfv_df.merge(user_id_data, how="left", left_on="user_delfi_id", right_on="delfi_id").rename(columns={"id": "user_id"})
rfv_df.head()

,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score,user_id,delfi_id
0,1735,8,9,2023-02-26,127.50,83.50,2,8.00,4362,1735
1,2140,20,33,2023-02-28,258.00,55.00,0,114.89,14449,2140
2,2320,19,37,2023-02-28,114.50,62.50,0,115.57,5856,2320
3,2872,10,20,2023-02-28,158.50,89.50,0,44.72,56979,2872
4,3069,25,86,2023-02-28,57.00,53.75,0,231.84,56747,3069


In [24]:
# Users who were active during the whole period.
users_query = """
    SELECT DISTINCT(P1.user_id)
    FROM (SELECT T.user_id, T.completion_timestamp, S.plan, US.renewal,
        ROW_NUMBER() OVER (PARTITION BY T.user_id ORDER BY T.completion_timestamp) AS row_num
        FROM paywall.transactions AS T
		LEFT JOIN paywall.user_subscriptions AS US
		ON T.user_id = US.user_id
		LEFT JOIN paywall.subscriptions AS S
		ON US.subscription_id = S.id
		WHERE US.service_id IN (1, 3, 4, 9, 10)
		AND S.included_in_count = 1
		AND T.completion_timestamp IS NOT NULL
		AND S.sf_category = "Paid subscriptions"
		AND US.valid_till > "%s 23:59:59") AS P1
    WHERE P1.row_num = 1
    AND P1.completion_timestamp < "2023-02-01 00:00:00"
    """ % latest_date

users = pd.read_sql(users_query, delfi_paywall_conn)
users = users.astype("string")

In [25]:
users_final_df = rfv_df.merge(users, on="user_id").drop(columns="delfi_id")
users_final_df = users_final_df.astype({"user_delfi_id": "string", "user_id": "string"})

unique_users = users_final_df["user_delfi_id"].nunique()
print(f"Final data consists of {unique_users} unique users.")

users_final_df.head()

Final data consists of 11439 unique users.


,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score,user_id
0,2140,20,33,2023-02-28,258.00,55.00,0,114.89,14449
1,2320,19,37,2023-02-28,114.50,62.50,0,115.57,5856
2,2872,10,20,2023-02-28,158.50,89.50,0,44.72,56979
3,3069,25,86,2023-02-28,57.00,53.75,0,231.84,56747
4,11878,9,11,2023-02-24,158.00,76.00,4,5.97,99419


In [26]:
recency_threshold = 6

syntetic_rfv_data = users_final_df.copy()
syntetic_rfv_data["user_type"] = np.where(users_final_df["recency"] <= recency_threshold, "fans", "sleepers")

pd.DataFrame({
    "Users": syntetic_rfv_data.groupby("user_type")["user_id"].count(),
    "Share": syntetic_rfv_data.groupby("user_type")["user_id"].count() / syntetic_rfv_data["user_id"].count() * 100,
    "RFV score": syntetic_rfv_data.groupby("user_type")["rfv_score"].median(),
    "Frequency": syntetic_rfv_data.groupby("user_type")["frequency"].median(),
    "Recency": syntetic_rfv_data.groupby("user_type")["recency"].median(),
    "Volume": syntetic_rfv_data.groupby("user_type")["volume"].median()
}).set_axis(["Fans: Recency equal or below 6", "Sleepers: Recency above 6"], axis="index")

,Users,Share,RFV score,Frequency,Recency,Volume
Fans: Recency equal or below 6,9452,82.63,29.15,12.00,0.00,18.00
Sleepers: Recency above 6,1987,17.37,0.32,2.00,13.00,3.00


In [27]:
frequency_threshold = syntetic_rfv_data[syntetic_rfv_data["recency"] == recency_threshold]["frequency"].median()
volume_threshold = syntetic_rfv_data[syntetic_rfv_data["recency"] == recency_threshold]["volume"].median()
print("Threshold of Frequency:", frequency_threshold)
print("Threshold of Volume:", volume_threshold)

Threshold of Frequency: 5.0
Threshold of Volume: 6.0


In [29]:
syntetic_rfv_score = (frequency_threshold * np.sqrt((volume_threshold))) / (recency_threshold + 1)
print("Threshold of RFV score:", syntetic_rfv_score)
print("Duration of period (Days):", initial_activity_data["date"].nunique())

Threshold of RFV score: 1.749635530559413
Duration of period (Days): 28
